# LeetCode 741 — Cherry Pickup bằng Python

- Link: https://leetcode.com/problems/cherry-pickup/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động trên lưới (grid DP), Hai người đi đồng thời, Giảm chiều trạng thái, Đệ quy có nhớ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def cherryPickup(self, grid: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Bạn đi từ góc trên-trái xuống góc dưới-phải để hái anh đào, rồi quay ngược về. Ô nào đã hái thì hết anh đào. Hỏi hái được nhiều nhất bao nhiêu?

Bài này nổi tiếng vì **lời giải tham lam "đi lượt đi tốt nhất, rồi lượt về tốt nhất" là SAI**. Mẹo then chốt là biến "đi rồi về" thành "**hai người cùng đi từ đầu đến cuối, cùng một lúc**" — sau đó dùng DP với trạng thái là vị trí của cả hai người.

Sau notebook này bạn sẽ:

1. Viết DP đường đi một người trên lưới (nền tảng).
2. Hiểu vì sao tham lam hai lượt thất bại và tự tìm phản ví dụ bằng máy.
3. Chuyển "đi rồi về" thành "hai người đi song song", xử lý trường hợp hai người đứng **cùng ô**.
4. Giảm trạng thái từ 4 chiều `(r1,c1,r2,c2)` xuống 3 chiều nhờ `r1 + c1 = r2 + c2`.
5. Viết DP bottom-up theo **số bước** `t` với bảng 2 chiều cuộn (`O(n^2)` bộ nhớ).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho lưới `grid` kích thước `n × n`, mỗi ô là:

- `0`: ô trống, đi qua được;
- `1`: ô có một quả anh đào, đi qua được (hái xong ô thành `0`);
- `-1`: ô có gai, **không** đi qua được.

Quy tắc:

1. Đi từ `(0, 0)` đến `(n-1, n-1)`, mỗi bước sang **phải** hoặc **xuống**, chỉ qua ô hợp lệ (`0` hoặc `1`).
2. Sau đó quay về `(0, 0)`, mỗi bước sang **trái** hoặc **lên**.
3. Đi qua ô có anh đào thì hái (ô thành `0`).
4. Nếu không có đường hợp lệ từ `(0,0)` đến `(n-1,n-1)` thì không hái được gì → trả `0`.

Trả về số anh đào tối đa.

**Ví dụ 1**

```text
grid = [[0, 1, -1],
        [1, 0, -1],
        [1, 1,  1]]
Output: 5
Lượt đi: xuống, xuống, phải, phải -> hái 4 quả; lưới còn [[0,1,-1],[0,0,-1],[0,0,0]].
Lượt về: trái, lên, lên, trái -> hái thêm 1 quả. Tổng 5.
```

**Ví dụ 2**

```text
grid = [[ 1, 1, -1],
        [ 1,-1,  1],
        [-1, 1,  1]]
Output: 0   (không có đường đi tới góc dưới-phải)
```

**Ràng buộc**

- `n == grid.length == grid[i].length`, `1 <= n <= 50`
- `grid[i][j]` ∈ `{-1, 0, 1}`
- `grid[0][0] != -1`, `grid[n-1][n-1] != -1`

In [1]:
from typing import List
EXAMPLES = [
    ([[0, 1, -1], [1, 0, -1], [1, 1, 1]], 5),
    ([[1, 1, -1], [1, -1, 1], [-1, 1, 1]], 0),
]
def show(grid):
    for row in grid:
        print(" ".join(f"{v:>2}" for v in row))
for g, e in EXAMPLES:
    show(g); print("-> mong đợi", e, "\n")

 0  1 -1
 1  0 -1
 1  1  1
-> mong đợi 5 

 1  1 -1
 1 -1  1
-1  1  1
-> mong đợi 0 



## 3. Kiến thức nền

### 3.1. DP đường đi một người: nhiều anh đào nhất

`best[r][c]` = số anh đào lớn nhất trên một đường từ `(0,0)` tới `(r,c)`. Công thức: `best[r][c] = grid[r][c] + max(best[r-1][c], best[r][c-1])`; ô gai hoặc không tới được = `-inf`.

In [2]:
NEG = float('-inf')

def best_single_path(grid):
    n = len(grid)
    best = [[NEG] * n for _ in range(n)]
    for r in range(n):
        for c in range(n):
            if grid[r][c] == -1:
                continue
            if r == 0 and c == 0:
                best[r][c] = grid[0][0]
                continue
            prev = max(best[r - 1][c] if r else NEG, best[r][c - 1] if c else NEG)
            if prev != NEG:
                best[r][c] = prev + grid[r][c]
    return best[n - 1][n - 1]

print(best_single_path(EXAMPLES[0][0]), best_single_path(EXAMPLES[1][0]))

4 -inf


### 3.2. `float('-inf')` làm "không tới được"

`-inf + 5 == -inf` và `max(-inf, 3) == 3` → rất tiện để lan truyền "không hợp lệ" mà không cần `if` khắp nơi. Cuối cùng nhớ đổi `-inf` thành `0` theo yêu cầu đề.

In [3]:
print(NEG + 5, max(NEG, 3), max(0, NEG))

-inf 3 0


### 3.3. Liệt kê mọi đường đi bằng `itertools.combinations`

Một đường từ `(0,0)` tới `(n-1,n-1)` gồm `2n-2` bước, trong đó đúng `n-1` bước xuống. Chọn vị trí các bước xuống ↔ `combinations(range(2n-2), n-1)`. Số đường = `C(2n-2, n-1)`.

In [4]:
from itertools import combinations
from math import comb

def all_paths(n):
    for downs in combinations(range(2 * n - 2), n - 1):
        r = c = 0
        cells = [(0, 0)]
        ds = set(downs)
        for step in range(2 * n - 2):
            if step in ds: r += 1
            else: c += 1
            cells.append((r, c))
        yield cells

print(list(all_paths(2)))
print("Số đường n=3:", comb(4, 2), " n=5:", comb(8, 4), " n=50:", comb(98, 49))

[[(0, 0), (1, 0), (1, 1)], [(0, 0), (0, 1), (1, 1)]]
Số đường n=3: 6  n=5: 70  n=50: 25477612258980856902730428600


### 3.4. `lru_cache` cho đệ quy nhiều tham số

In [5]:
from functools import lru_cache
import sys
sys.setrecursionlimit(10000)

@lru_cache(None)
def grid_paths(r, c):
    if r == 0 or c == 0:
        return 1
    return grid_paths(r - 1, c) + grid_paths(r, c - 1)

print(grid_paths(4, 4), comb(8, 4))

70 70


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Lượt về = một lượt đi khác

Đi ngược từ `(n-1,n-1)` về `(0,0)` bằng trái/lên chính là một đường **đi xuôi** (phải/xuống) đọc ngược lại. Vậy bài toán là: chọn **hai** đường xuôi `P` và `Q` để tổng anh đào trên **hợp** `P ∪ Q` lớn nhất (ô chung chỉ hái một lần).

### 4.2. Tính tay ví dụ 1

```text
 0  1 -1
 1  0 -1
 1  1  1
```

- Đường `P`: (0,0)→(1,0)→(2,0)→(2,1)→(2,2): hái 1+1+1+1 = 4.
- Đường `Q`: (0,0)→(0,1)→(1,1)→(2,1)→(2,2): ô mới có anh đào là (0,1) → +1.
- Hợp: 5 ô có anh đào. Tổng 5. ✔ (Tổng số anh đào cả lưới là 5, không thể hơn.)

### 4.3. Vì sao "tham lam hai lượt" có thể sai?

Lượt đi tốt nhất có thể "ăn" mất những ô nằm ở cả hai vùng, khiến lượt về không còn gì. Một cặp đường **phối hợp** (mỗi người lo một vùng) có thể tốt hơn, dù riêng lượt đi không phải tối ưu. Ta sẽ cho máy tìm phản ví dụ ở mục 6.

In [6]:
g = EXAMPLES[0][0]
P = [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]
Q = [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2)]
print("Tổng trên hợp P ∪ Q:", sum(g[r][c] for r, c in set(P) | set(Q)))

Tổng trên hợp P ∪ Q: 5


## 5. Cách 1 — Brute Force: thử mọi lượt đi, DP lượt về

**Ý tưởng.** Với mỗi đường đi hợp lệ `P` (không qua gai): hái anh đào trên `P`, đặt các ô đó về `0`, rồi tính lượt về tốt nhất bằng DP một người (mục 3.1). Lấy max.

Đây là brute force **đúng tuyệt đối** (thử mọi `P`, và với mỗi `P` thì lượt về tối ưu), dùng làm trọng tài.

In [7]:
def cherry_brute(grid):
    n = len(grid)
    best = NEG
    for cells in all_paths(n):
        if any(grid[r][c] == -1 for r, c in cells):
            continue
        got = sum(grid[r][c] for r, c in set(cells))
        g2 = [row[:] for row in grid]
        for r, c in cells:
            g2[r][c] = 0
        back = best_single_path(g2)
        best = max(best, got + back)
    return max(0, best) if best != NEG else 0

for g, e in EXAMPLES:
    print(cherry_brute(g), cherry_brute(g) == e)

5 True
0 True


**Độ phức tạp.** `C(2n-2, n-1)` đường × `O(n^2)` DP. Với `n = 50` là ~`2.5·10^28` đường — bất khả thi. Chỉ dùng cho `n ≤ 7`.

**Nút thắt.** Ta "chốt" toàn bộ lượt đi rồi mới xét lượt về. Cách hiệu quả là cho **hai người đi cùng lúc** và chỉ nhớ **vị trí hiện tại** của họ — đủ để biết ô nào bị hái trùng.

## 6. Bước đột phá

### 6.1. Tham lam hai lượt sai — để máy tìm phản ví dụ

In [8]:
import random

def best_path_cells(grid):
    """Trả về (giá trị, danh sách ô) của đường một người tốt nhất, hoặc (NEG, None)."""
    n = len(grid)
    best = [[NEG] * n for _ in range(n)]
    par = [[None] * n for _ in range(n)]
    for r in range(n):
        for c in range(n):
            if grid[r][c] == -1:
                continue
            if r == 0 and c == 0:
                best[0][0] = grid[0][0]; continue
            up = best[r - 1][c] if r else NEG
            left = best[r][c - 1] if c else NEG
            if max(up, left) == NEG:
                continue
            if up >= left:
                best[r][c], par[r][c] = up + grid[r][c], (r - 1, c)
            else:
                best[r][c], par[r][c] = left + grid[r][c], (r, c - 1)
    if best[n - 1][n - 1] == NEG:
        return NEG, None
    cells, cur = [], (n - 1, n - 1)
    while cur:
        cells.append(cur); cur = par[cur[0]][cur[1]]
    return best[n - 1][n - 1], cells[::-1]

def cherry_greedy_wrong(grid):
    v1, cells = best_path_cells(grid)
    if cells is None:
        return 0
    g2 = [row[:] for row in grid]
    for r, c in cells:
        g2[r][c] = 0
    return v1 + best_single_path(g2)

rng = random.Random(741)
found = None
for _ in range(5000):
    n = rng.randint(3, 5)
    g = [[rng.choice([0, 1, 1, -1]) for _ in range(n)] for _ in range(n)]
    g[0][0] = g[n - 1][n - 1] = 0
    if cherry_greedy_wrong(g) != cherry_brute(g):
        found = g
        break
show(found)
print("tham lam:", cherry_greedy_wrong(found), " tối ưu:", cherry_brute(found))

 0  0  0  1
 0 -1  0  1
 1  1  1  1
-1 -1  1  0
tham lam: 6  tối ưu: 7


### 6.2. Hai người đi đồng thời

**Insight 1.** Thay "đi rồi về" bằng hai người A và B **cùng xuất phát** từ `(0,0)`, mỗi giây cả hai cùng bước một bước (phải hoặc xuống), cùng tới `(n-1,n-1)` sau `2n-2` giây.

**Insight 2 (hái trùng).** Hai đường chỉ có thể đi qua **cùng một ô** vào **cùng một thời điểm** `t`: ô `(r,c)` luôn được đến ở bước `t = r + c`. Vậy chỉ cần kiểm tra "hai người có đang đứng cùng ô không" tại mỗi bước — nếu có, chỉ cộng anh đào một lần.

**Insight 3 (giảm chiều).** Tại bước `t`: `c1 = t - r1`, `c2 = t - r2`. Trạng thái `(r1, c1, r2, c2)` thu về `(t, r1, r2)` → `O(n^3)` trạng thái, mỗi trạng thái 4 chuyển (A xuống/phải × B xuống/phải).

**Công thức.**

```text
dp[t][r1][r2] = gain(t, r1, r2) + max(dp[t-1][r1'][r2'])  với r1' ∈ {r1, r1-1}, r2' ∈ {r2, r2-1}
gain = grid[r1][c1] + (grid[r2][c2] nếu r1 != r2, ngược lại 0)
ô gai / ra ngoài lưới -> -inf
```

**Quy tắc nếu… thì…**

- Nếu một trong hai người đứng trên gai hoặc ngoài lưới → trạng thái `-inf`.
- Nếu `r1 == r2` (cùng bước ⇒ cùng ô) → chỉ cộng một lần.
- Đối xứng: có thể giả sử `r1 ≤ r2` để giảm một nửa trạng thái (tuỳ chọn).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ 4 chiều `(r1, c1, r2, c2)`

**Giải thích.** `f(r1,c1,r2,c2)` = số anh đào tối đa hái được từ trạng thái hiện tại đến đích, khi A ở `(r1,c1)` và B ở `(r2,c2)` (cùng số bước). Cài trực tiếp; chưa tận dụng `r1+c1 = r2+c2` để giảm chiều — nhưng `lru_cache` chỉ lưu các trạng thái thực sự gặp, và đó **chỉ** là các trạng thái cùng bước, nên thực tế vẫn là `O(n^3)` trạng thái.

In [9]:
def cherry_memo4(grid):
    n = len(grid)

    @lru_cache(None)
    def f(r1, c1, r2, c2):
        if r1 >= n or c1 >= n or r2 >= n or c2 >= n:
            return NEG
        if grid[r1][c1] == -1 or grid[r2][c2] == -1:
            return NEG
        gain = grid[r1][c1] + (grid[r2][c2] if (r1, c1) != (r2, c2) else 0)
        if r1 == n - 1 and c1 == n - 1:
            return gain                      # B cũng đã tới đích (cùng số bước)
        nxt = max(f(r1 + 1, c1, r2 + 1, c2), f(r1 + 1, c1, r2, c2 + 1),
                  f(r1, c1 + 1, r2 + 1, c2), f(r1, c1 + 1, r2, c2 + 1))
        return gain + nxt

    return max(0, f(0, 0, 0, 0))

for g, e in EXAMPLES:
    print(cherry_memo4(g), cherry_memo4(g) == e)

5 True
0 True


**Code nộp LeetCode (Cách 2).**

```python
import sys
from functools import lru_cache

class Solution:
    def cherryPickup(self, grid: List[List[int]]) -> int:
        sys.setrecursionlimit(10000)
        n = len(grid)
        NEG = float('-inf')

        @lru_cache(None)
        def f(r1, c1, r2, c2):
            if r1 >= n or c1 >= n or r2 >= n or c2 >= n:
                return NEG
            if grid[r1][c1] == -1 or grid[r2][c2] == -1:
                return NEG
            gain = grid[r1][c1] + (grid[r2][c2] if (r1, c1) != (r2, c2) else 0)
            if r1 == n - 1 and c1 == n - 1:
                return gain
            return gain + max(f(r1 + 1, c1, r2 + 1, c2), f(r1 + 1, c1, r2, c2 + 1),
                              f(r1, c1 + 1, r2 + 1, c2), f(r1, c1 + 1, r2, c2 + 1))

        return max(0, f(0, 0, 0, 0))
```

**Độ phức tạp.** `O(n^3)` trạng thái thực sự gặp, mỗi trạng thái `O(1)` → `O(n^3)` time, `O(n^3)` space (`50^3 = 125000`). Khoá 4 số làm cache nặng hơn Cách 3.

### 7.2. Cách 3 — Đệ quy có nhớ 3 chiều `(r1, c1, r2)`

**Giải thích.** Bỏ `c2` vì `c2 = r1 + c1 - r2`. Phần còn lại giống hệt Cách 2.

In [10]:
def cherry_memo3(grid):
    n = len(grid)

    @lru_cache(None)
    def f(r1, c1, r2):
        c2 = r1 + c1 - r2
        if r1 >= n or c1 >= n or r2 >= n or c2 >= n:
            return NEG
        if grid[r1][c1] == -1 or grid[r2][c2] == -1:
            return NEG
        gain = grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
        if r1 == n - 1 and c1 == n - 1:
            return gain
        return gain + max(f(r1 + 1, c1, r2 + 1), f(r1 + 1, c1, r2),
                          f(r1, c1 + 1, r2 + 1), f(r1, c1 + 1, r2))

    return max(0, f(0, 0, 0))

for g, e in EXAMPLES:
    print(cherry_memo3(g), cherry_memo3(g) == e)

5 True
0 True


> `f(r1 + 1, c1, r2)` nghĩa là A xuống, B **sang phải** (vì `r2` giữ nguyên còn tổng bước tăng 1 nên `c2` tự tăng 1).

**Code nộp LeetCode (Cách 3).**

```python
import sys
from functools import lru_cache

class Solution:
    def cherryPickup(self, grid: List[List[int]]) -> int:
        sys.setrecursionlimit(10000)
        n = len(grid)
        NEG = float('-inf')

        @lru_cache(None)
        def f(r1, c1, r2):
            c2 = r1 + c1 - r2
            if r1 >= n or c1 >= n or r2 >= n or c2 >= n:
                return NEG
            if grid[r1][c1] == -1 or grid[r2][c2] == -1:
                return NEG
            gain = grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
            if r1 == n - 1 and c1 == n - 1:
                return gain
            return gain + max(f(r1 + 1, c1, r2 + 1), f(r1 + 1, c1, r2),
                              f(r1, c1 + 1, r2 + 1), f(r1, c1 + 1, r2))

        return max(0, f(0, 0, 0))
```

**Độ phức tạp.** `O(n^3)` time và space. Độ sâu đệ quy `2n - 1 ≤ 99`.

### 7.3. Cách 4 — Bottom-up theo số bước `t`, bảng 2 chiều cuộn

**Giải thích.** Chỉ cần bảng của bước `t-1` để tính bước `t` → giữ `dp[r1][r2]` (`n × n`) và tạo bảng mới mỗi bước. Với mỗi `t` từ `1` đến `2n-2`, `r1`, `r2` chạy trong `[max(0, t-n+1), min(n-1, t)]` để `c = t - r` hợp lệ.

In [11]:
def cherry_bottom_up(grid):
    n = len(grid)
    dp = [[NEG] * n for _ in range(n)]
    dp[0][0] = grid[0][0]
    for t in range(1, 2 * n - 1):
        new = [[NEG] * n for _ in range(n)]
        lo, hi = max(0, t - n + 1), min(n - 1, t)
        for r1 in range(lo, hi + 1):
            c1 = t - r1
            if grid[r1][c1] == -1:
                continue
            for r2 in range(r1, hi + 1):              # đối xứng: r1 <= r2
                c2 = t - r2
                if grid[r2][c2] == -1:
                    continue
                prev = dp[r1][r2]
                if r1:
                    prev = max(prev, dp[r1 - 1][r2])
                if r2:
                    prev = max(prev, dp[r1][r2 - 1])
                if r1 and r2:
                    prev = max(prev, dp[r1 - 1][r2 - 1])
                if prev == NEG:
                    continue
                gain = grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
                new[r1][r2] = prev + gain
        dp = new
    return max(0, dp[n - 1][n - 1])

for g, e in EXAMPLES:
    print(cherry_bottom_up(g), cherry_bottom_up(g) == e)

5 True
0 True


> Vì sao đối xứng `r1 ≤ r2` vẫn đúng khi tra `dp[r1-1][r2]` hay `dp[r1][r2-1]`? Khi `r1 ≤ r2`: `r1 - 1 ≤ r2` ✔; `r1 ≤ r2 - 1` chỉ sai khi `r1 == r2`, lúc đó `dp[r1][r2-1]` có `r1 > r2 - 1` → ô "dưới đường chéo" mang `-inf`. Nhưng trạng thái `(r1, r2-1)` với `r1 > r2-1` tương đương `(r2-1, r1)` = `dp[r1-1][r2]` (vì `r1 = r2`) — đã được xét bằng nhánh `dp[r1-1][r2]`. Nên không mất gì.

**Code nộp LeetCode (Cách 4).**

```python
class Solution:
    def cherryPickup(self, grid: List[List[int]]) -> int:
        n = len(grid)
        NEG = float('-inf')
        dp = [[NEG] * n for _ in range(n)]
        dp[0][0] = grid[0][0]
        for t in range(1, 2 * n - 1):
            new = [[NEG] * n for _ in range(n)]
            lo, hi = max(0, t - n + 1), min(n - 1, t)
            for r1 in range(lo, hi + 1):
                c1 = t - r1
                if grid[r1][c1] == -1:
                    continue
                for r2 in range(r1, hi + 1):
                    c2 = t - r2
                    if grid[r2][c2] == -1:
                        continue
                    prev = dp[r1][r2]
                    if r1:
                        prev = max(prev, dp[r1 - 1][r2])
                    if r2:
                        prev = max(prev, dp[r1][r2 - 1])
                    if r1 and r2:
                        prev = max(prev, dp[r1 - 1][r2 - 1])
                    if prev == NEG:
                        continue
                    new[r1][r2] = prev + grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
            dp = new
        return max(0, dp[n - 1][n - 1])
```

**Độ phức tạp.** `O(n^3)` time (khoảng `(2n) · n^2 / 2`), `O(n^2)` space.

## 8. TRACE từng bước

### 8.1. Bảng `dp[r1][r2]` theo `t` cho ví dụ 1

Lưới:

```text
 0  1 -1
 1  0 -1
 1  1  1
```

| t | Ô hợp lệ trên đường chéo `r+c=t` | dp (r1 ≤ r2) |
|---|---|---|
| 0 | (0,0) | dp[0][0] = 0 |
| 1 | (0,1)=1, (1,0)=1 | [0][0]=1, [0][1]=2, [1][1]=1 |
| 2 | (0,2)=gai, (1,1)=0, (2,0)=1 | [1][1]=2, [1][2]=3, [2][2]=2 |
| 3 | (1,2)=gai, (2,1)=1 | [2][2]=4 |
| 4 | (2,2)=1 | [2][2]=5 |

In [12]:
def trace_bottom_up(grid):
    n = len(grid)
    dp = [[NEG] * n for _ in range(n)]
    dp[0][0] = grid[0][0]
    print("t=0: dp[0][0] =", dp[0][0])
    for t in range(1, 2 * n - 1):
        new = [[NEG] * n for _ in range(n)]
        lo, hi = max(0, t - n + 1), min(n - 1, t)
        for r1 in range(lo, hi + 1):
            c1 = t - r1
            if grid[r1][c1] == -1: continue
            for r2 in range(r1, hi + 1):
                c2 = t - r2
                if grid[r2][c2] == -1: continue
                cands = [dp[r1][r2]]
                if r1: cands.append(dp[r1 - 1][r2])
                if r2: cands.append(dp[r1][r2 - 1])
                if r1 and r2: cands.append(dp[r1 - 1][r2 - 1])
                prev = max(cands)
                if prev == NEG: continue
                gain = grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
                new[r1][r2] = prev + gain
        dp = new
        cells = {f"A({r1},{t-r1}) B({r2},{t-r2})": dp[r1][r2]
                 for r1 in range(n) for r2 in range(n) if dp[r1][r2] != NEG}
        print(f"t={t}:", cells)
    return max(0, dp[n - 1][n - 1])

print("Kết quả:", trace_bottom_up(EXAMPLES[0][0]))

t=0: dp[0][0] = 0
t=1: {'A(0,1) B(0,1)': 1, 'A(0,1) B(1,0)': 2, 'A(1,0) B(1,0)': 1}
t=2: {'A(1,1) B(1,1)': 2, 'A(1,1) B(2,0)': 3, 'A(2,0) B(2,0)': 2}
t=3: {'A(2,1) B(2,1)': 4}
t=4: {'A(2,2) B(2,2)': 5}
Kết quả: 5


### 8.2. Khôi phục hai đường đi tối ưu

Lưu thêm "trạng thái cha" để in ra hai đường A và B — giúp tin rằng đáp án là thật.

In [13]:
def cherry_paths(grid):
    n = len(grid)

    @lru_cache(None)
    def f(r1, c1, r2):
        c2 = r1 + c1 - r2
        if r1 >= n or c1 >= n or r2 >= n or c2 >= n or grid[r1][c1] == -1 or grid[r2][c2] == -1:
            return NEG
        gain = grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
        if r1 == n - 1 and c1 == n - 1:
            return gain
        return gain + max(f(r1 + 1, c1, r2 + 1), f(r1 + 1, c1, r2), f(r1, c1 + 1, r2 + 1), f(r1, c1 + 1, r2))

    if f(0, 0, 0) == NEG:
        return 0, None, None
    A, B = [(0, 0)], [(0, 0)]
    r1 = c1 = r2 = 0
    while (r1, c1) != (n - 1, n - 1):
        moves = [(r1 + 1, c1, r2 + 1), (r1 + 1, c1, r2), (r1, c1 + 1, r2 + 1), (r1, c1 + 1, r2)]
        r1, c1, r2 = max(moves, key=lambda m: f(*m))
        A.append((r1, c1)); B.append((r2, r1 + c1 - r2))
    return f(0, 0, 0), A, B

val, A, B = cherry_paths(found)
show(found)
print("tối ưu =", val)
print("A:", A)
print("B:", B)

 0  0  0  1
 0 -1  0  1
 1  1  1  1
-1 -1  1  0
tối ưu = 7
A: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (3, 3)]
B: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (3, 3)]


## 9. Code nộp LeetCode

Bản chính: **Cách 4 (bottom-up, bộ nhớ `O(n^2)`)**.

In [14]:
class Solution:
    def cherryPickup(self, grid: List[List[int]]) -> int:
        n = len(grid)
        NEG = float('-inf')
        dp = [[NEG] * n for _ in range(n)]
        dp[0][0] = grid[0][0]
        for t in range(1, 2 * n - 1):
            new = [[NEG] * n for _ in range(n)]
            lo, hi = max(0, t - n + 1), min(n - 1, t)
            for r1 in range(lo, hi + 1):
                c1 = t - r1
                if grid[r1][c1] == -1:
                    continue
                for r2 in range(r1, hi + 1):
                    c2 = t - r2
                    if grid[r2][c2] == -1:
                        continue
                    prev = dp[r1][r2]
                    if r1:
                        prev = max(prev, dp[r1 - 1][r2])
                    if r2:
                        prev = max(prev, dp[r1][r2 - 1])
                    if r1 and r2:
                        prev = max(prev, dp[r1 - 1][r2 - 1])
                    if prev == NEG:
                        continue
                    new[r1][r2] = prev + grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
            dp = new
        return max(0, dp[n - 1][n - 1])

for g, e in EXAMPLES:
    assert Solution().cherryPickup(g) == e
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao thuật toán đúng

**Bước 1 — quy về hai đường xuôi.** Lượt về (trái/lên) đảo ngược là một đường xuôi. Anh đào thu được = tổng `grid` trên `P ∪ Q` (ô chung chỉ một lần). Ngược lại, mọi cặp đường xuôi hợp lệ `(P, Q)` cho một chuyến đi–về hợp lệ. → Đáp án = `max_{P,Q} Σ_{(r,c) ∈ P∪Q} grid[r][c]`.

**Bước 2 — đồng bộ thời gian.** Trên đường xuôi, ô `(r,c)` luôn là ô thứ `r + c` (tính từ 0). Vì thế `P` và `Q` chung ô `(r,c)` ⇔ tại bước `t = r+c` cả hai cùng ở `(r,c)` ⇔ `r1 == r2` tại bước đó. Tổng trên hợp = `Σ_t gain(t)` với `gain` như công thức. → Bài toán tách được theo từng bước `t`.

**Bước 3 — cấu trúc con tối ưu.** Giá trị tốt nhất để tới trạng thái `(t, r1, r2)` = `gain(t, r1, r2)` + max giá trị tốt nhất của 4 trạng thái tiền nhiệm ở bước `t-1` (vì `gain` chỉ phụ thuộc vị trí hiện tại, không phụ thuộc lịch sử). Quy nạp theo `t` → `dp` đúng.

**Bước 4 — đối xứng.** Đổi vai A và B không đổi giá trị → `dp[r1][r2] = dp[r2][r1]`; chỉ cần tính `r1 ≤ r2` (giải thích ở cuối 7.3).

**Không có đường.** Nếu `(n-1,n-1)` không tới được, mọi trạng thái cuối là `-inf` → trả `0`.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(C(2n-2,n-1) · n^2)` | `O(n^2)` |
| Tham lam 2 lượt (SAI) | `O(n^2)` | `O(n^2)` |
| 2. Memo 4 chiều | `O(n^3)` trạng thái gặp | `O(n^3)` |
| 3. Memo 3 chiều | `O(n^3)` | `O(n^3)` |
| 4. Bottom-up cuộn | `O(n^3)` (≈ `n^3` phép) | `O(n^2)` |

Với `n = 50`: `n^3 = 125000` trạng thái — rất nhanh.

In [15]:
import time
rng = random.Random(50)
n = 50
big = [[rng.choice([0, 1, 1, 1, -1]) for _ in range(n)] for _ in range(n)]
big[0][0] = big[n - 1][n - 1] = 1
for fn in [cherry_memo4, cherry_memo3, cherry_bottom_up, Solution().cherryPickup]:
    t = time.perf_counter(); r = fn(big); t = time.perf_counter() - t
    print(f"{getattr(fn, '__name__', ''):<18} {t*1000:8.1f} ms -> {r}")

cherry_memo4           11.5 ms -> 173
cherry_memo3            9.9 ms -> 173
cherry_bottom_up        5.9 ms -> 173
cherryPickup            5.7 ms -> 173


## 12. Bộ test

In [16]:
ALL = [cherry_brute, cherry_memo4, cherry_memo3, cherry_bottom_up, Solution().cherryPickup]

EDGE = [
    ([[0]], 0),
    ([[1]], 1),
    ([[1, 1], [1, 1]], 4),
    ([[1, -1], [-1, 1]], 0),                 # bị chặn
    ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 8),  # hai đường phủ 8/9 ô
    ([[0, 0, 0], [0, -1, 0], [0, 0, 0]], 0),
    ([[1, 1, 1, 1], [-1, -1, -1, 1], [1, 1, 1, 1], [1, -1, -1, -1]], 0),
]
for fn in ALL:
    for g, e in EXAMPLES + EDGE:
        assert fn(g) == e, (fn, g, fn(g), e)
print("Ví dụ đề + edge case OK")

Ví dụ đề + edge case OK


In [17]:
rng = random.Random(7410)
for _ in range(1500):
    n = rng.randint(1, 5)
    g = [[rng.choice([0, 1, 1, -1]) for _ in range(n)] for _ in range(n)]
    g[0][0] = rng.choice([0, 1]); g[n - 1][n - 1] = rng.choice([0, 1])
    ref = cherry_brute(g)
    for fn in ALL[1:]:
        assert fn(g) == ref, (fn, g, fn(g), ref)
print("1500 lưới ngẫu nhiên: mọi cách khớp brute force")

1500 lưới ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam hai lượt

**Code sai:** `cherry_greedy_wrong` ở mục 6.1.

**Vì sao sai:** lượt đi tối ưu riêng lẻ có thể phá hỏng lượt về. Phản ví dụ máy tìm được đã in ở 6.1.

**Code đúng:** DP hai người đồng thời.

In [18]:
print("sai:", cherry_greedy_wrong(found), " đúng:", Solution().cherryPickup(found))

sai: 6  đúng: 7


### Lỗi 2 — Cộng anh đào hai lần khi hai người cùng ô

**Code sai:** `gain = grid[r1][c1] + grid[r2][c2]` luôn luôn.

**Vì sao sai:** `[[1]]` sẽ ra 2; lưới toàn `1` sẽ vượt tổng số anh đào.

**Code đúng:** `+ (grid[r2][c2] if r1 != r2 else 0)`.

In [19]:
def double_count_wrong(grid):
    n = len(grid)
    @lru_cache(None)
    def f(r1, c1, r2):
        c2 = r1 + c1 - r2
        if r1 >= n or c1 >= n or r2 >= n or c2 >= n or grid[r1][c1] == -1 or grid[r2][c2] == -1:
            return NEG
        gain = grid[r1][c1] + grid[r2][c2]                    # SAI
        if r1 == n - 1 and c1 == n - 1:
            return gain
        return gain + max(f(r1 + 1, c1, r2 + 1), f(r1 + 1, c1, r2), f(r1, c1 + 1, r2 + 1), f(r1, c1 + 1, r2))
    return max(0, f(0, 0, 0))

print("sai:", double_count_wrong([[1]]), double_count_wrong([[1, 1], [1, 1]]),
      " đúng:", Solution().cherryPickup([[1]]), Solution().cherryPickup([[1, 1], [1, 1]]))

sai: 2 6  đúng: 1 4


### Lỗi 3 — Không phân biệt "không có đường" và "0 anh đào"

**Code sai:** dùng `0` cho ô gai/không tới được thay vì `-inf`.

**Vì sao sai:** đường "xuyên gai" được coi như hợp lệ với giá trị 0 + phần còn lại → có thể ra số dương khi thực ra không có đường (ví dụ 2 đề).

**Code đúng:** `-inf` cho trạng thái không hợp lệ, cuối cùng `max(0, ...)`.

In [20]:
def zero_instead_of_neg_wrong(grid):
    n = len(grid)
    @lru_cache(None)
    def f(r1, c1, r2):
        c2 = r1 + c1 - r2
        if r1 >= n or c1 >= n or r2 >= n or c2 >= n:
            return NEG
        if grid[r1][c1] == -1 or grid[r2][c2] == -1:
            return 0                                          # SAI
        gain = grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
        if r1 == n - 1 and c1 == n - 1:
            return gain
        return gain + max(f(r1 + 1, c1, r2 + 1), f(r1 + 1, c1, r2), f(r1, c1 + 1, r2 + 1), f(r1, c1 + 1, r2))
    return max(0, f(0, 0, 0))

g = EXAMPLES[1][0]
print("sai:", zero_instead_of_neg_wrong(g), " đúng:", Solution().cherryPickup(g))

sai: 3  đúng: 0


### Lỗi 4 — Cho hai người đi lệch bước (trạng thái `(r1,c1,r2,c2)` tự do)

**Code sai:** cho A đi trọn đường rồi B mới đi, nhưng lại dùng điều kiện "cùng ô" tại cùng lời gọi để trừ trùng.

**Vì sao sai:** nếu hai người không đi đồng bộ, cùng một ô có thể bị cả hai thăm **ở hai thời điểm khác nhau** — kiểm tra `(r1,c1) == (r2,c2)` tại cùng một trạng thái không bắt được.

**Code đúng:** luôn di chuyển **cả hai** mỗi bước (mọi chuyển trạng thái đều tăng cả `r1+c1` lẫn `r2+c2` thêm 1).

## 14. Mẹo & nhận dạng pattern

- **"Đi rồi về" / "hai người cùng đi" trên lưới** → DP hai người đồng thời, trạng thái `(bước, r1, r2)`.
- **Ô được đến ở bước `r + c`** → đồng bộ thời gian giúp phát hiện trùng ô dễ dàng.
- **Ràng buộc tuyến tính giữa các toạ độ** (`r1+c1 = r2+c2`) → bỏ bớt một chiều.
- **Tham lam theo từng phần thường sai** khi các phần dùng chung tài nguyên → thử phản ví dụ bằng brute force nhỏ.
- Bài anh em: *Cherry Pickup II (1463)* — hai robot cùng đi xuống từng hàng, trạng thái `(hàng, c1, c2)`.

## 15. Biến thể

1. **Cherry Pickup II (1463)**: hai robot từ hai góc trên, mỗi bước xuống một hàng và lệch cột `-1/0/+1`.
2. **Lưới chữ nhật `m × n`**: bước `t` chạy tới `m + n - 2`; `c = t - r` phải trong `[0, n)`.
3. **Ba người** đi đồng thời: trạng thái `(t, r1, r2, r3)` → `O(n^4)`; cộng ô trùng chỉ một lần.
4. **Mỗi ô có số anh đào bất kỳ** (không chỉ 0/1): công thức giữ nguyên.
5. **Chỉ một lượt** (không quay về): DP một người (mục 3.1).

In [21]:
# Biến thể 1: Cherry Pickup II (LeetCode 1463)
def cherry_pickup_ii(grid):
    rows, cols = len(grid), len(grid[0])
    dp = {(0, cols - 1): grid[0][0] + (grid[0][cols - 1] if cols > 1 else 0)}
    for r in range(1, rows):
        new = {}
        for (c1, c2), v in dp.items():
            for d1 in (-1, 0, 1):
                for d2 in (-1, 0, 1):
                    a, b = c1 + d1, c2 + d2
                    if 0 <= a < cols and 0 <= b < cols:
                        gain = grid[r][a] + (grid[r][b] if a != b else 0)
                        if v + gain > new.get((a, b), -1):
                            new[(a, b)] = v + gain
        dp = new
    return max(dp.values())

print(cherry_pickup_ii([[3, 1, 1], [2, 5, 1], [1, 5, 5], [2, 1, 1]]))          # 24
print(cherry_pickup_ii([[1, 0, 0, 0, 0, 0, 1], [2, 0, 0, 0, 0, 3, 0], [2, 0, 9, 0, 0, 0, 0],
                        [0, 3, 0, 5, 4, 0, 0], [1, 0, 2, 3, 0, 0, 6]]))        # 28

24
28


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mọi lượt đi + DP lượt về | mũ | `O(n^2)` | Trọng tài `n ≤ 6` |
| (Tham lam) | Lượt đi tốt nhất rồi lượt về | `O(n^2)` | `O(n^2)` | **Sai** — chỉ để học |
| 2. Memo 4 chiều | Hai người đồng thời, trạng thái đầy đủ | `O(n^3)` | `O(n^3)` | Viết nhanh, dễ hiểu |
| 3. Memo 3 chiều | Bỏ `c2` | `O(n^3)` | `O(n^3)` | Gọn, phỏng vấn |
| 4. Bottom-up cuộn | Bảng `n×n` theo bước `t`, đối xứng | `O(n^3)` | `O(n^2)` | Bản nộp tối ưu bộ nhớ |

## 17. Bài tập

**Bài 1 (đoán output).** `cherryPickup([[1,1,0],[0,1,1],[1,1,1]])` = ?

**Bài 2 (tính tay).** Lập bảng `dp` theo `t` (như 8.1) cho `[[1,1],[1,1]]`.

**Bài 3 (điền TODO).** Hoàn thiện `gain(grid, t, r1, r2)` trả về số anh đào hái được tại bước `t` (giả sử hai ô hợp lệ).

**Bài 4 (tự viết lại).** Viết lại Cách 3 (memo 3 chiều) trong hàm `my_cherry(grid)`.

**Bài 5 (mini challenge).** Mở rộng cho lưới **chữ nhật** `m × n`: viết `cherry_rect(grid)`.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def gain(grid, t, r1, r2):
    # TODO: c1 = t - r1; c2 = t - r2; cộng grid[r1][c1], cộng grid[r2][c2] nếu r1 != r2
    pass

# Bài 4 — TODO
def my_cherry(grid):
    # TODO: @lru_cache f(r1, c1, r2) như Cách 3
    pass

# Bài 5 — TODO
def cherry_rect(grid):
    # TODO: m, n = len(grid), len(grid[0]); điều kiện biên c < n, r < m; đích (m-1, n-1)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tổng số anh đào trong lưới là 7.
- Mức 2: thử cho A đi theo "đường trên" (0,0)→(0,1)→(1,1)→(1,2)→(2,2), B đi "đường dưới" (0,0)→(1,0)→(2,0)→(2,1)→(2,2).
- Mức 3: hợp hai đường chỉ bỏ sót ô (0,2) (giá trị 0) → hái hết **7**.

**Bài 2.**
- Mức 1: `t=0`: `dp[0][0] = 1`.
- Mức 2: `t=1`: `dp[0][0] = 2` (cùng ô (0,1)), `dp[0][1] = 3`, `dp[1][1] = 2`.
- Mức 3: `t=2`: `dp[1][1] = 3 + 1 = 4`.

**Bài 3.**
- Mức 1: tính `c1`, `c2` từ `t`.
- Mức 2: so sánh `r1 != r2` (cùng `t` nên cùng hàng ⇒ cùng ô).
- Mức 3: trả tổng.

**Bài 4.**
- Mức 1: `c2 = r1 + c1 - r2`.
- Mức 2: 4 lời gọi con.
- Mức 3: đích: `r1 == n-1 and c1 == n-1`; cuối cùng `max(0, ...)`.

**Bài 5.**
- Mức 1: biên: `r < m`, `c < n`.
- Mức 2: đích `(m-1, n-1)`.
- Mức 3: kiểm tra bằng brute force đã chỉnh cho lưới chữ nhật.

## 19. Đáp án tham khảo

In [23]:
# Bài 1, 2
assert Solution().cherryPickup([[1, 1, 0], [0, 1, 1], [1, 1, 1]]) == 7
assert Solution().cherryPickup([[1, 1], [1, 1]]) == 4

# Bài 3
def gain_ans(grid, t, r1, r2):
    c1, c2 = t - r1, t - r2
    return grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)

assert gain_ans([[1, 1], [1, 1]], 1, 0, 1) == 2
assert gain_ans([[1, 1], [1, 1]], 1, 0, 0) == 1
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_cherry_ans(grid):
    n = len(grid)
    @lru_cache(None)
    def f(r1, c1, r2):
        c2 = r1 + c1 - r2
        if max(r1, c1, r2, c2) >= n or grid[r1][c1] == -1 or grid[r2][c2] == -1:
            return NEG
        g = grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
        if r1 == c1 == n - 1:
            return g
        return g + max(f(r1 + 1, c1, r2 + 1), f(r1 + 1, c1, r2), f(r1, c1 + 1, r2 + 1), f(r1, c1 + 1, r2))
    return max(0, f(0, 0, 0))

# Bài 5
def cherry_rect_ans(grid):
    m, n = len(grid), len(grid[0])
    @lru_cache(None)
    def f(r1, c1, r2):
        c2 = r1 + c1 - r2
        if r1 >= m or r2 >= m or c1 >= n or c2 >= n or c2 < 0:
            return NEG
        if grid[r1][c1] == -1 or grid[r2][c2] == -1:
            return NEG
        g = grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
        if r1 == m - 1 and c1 == n - 1:
            return g
        return g + max(f(r1 + 1, c1, r2 + 1), f(r1 + 1, c1, r2), f(r1, c1 + 1, r2 + 1), f(r1, c1 + 1, r2))
    return max(0, f(0, 0, 0))

def rect_brute(grid):
    m, n = len(grid), len(grid[0])
    def paths():
        for downs in combinations(range(m + n - 2), m - 1):
            r = c = 0; cells = [(0, 0)]; ds = set(downs)
            for s in range(m + n - 2):
                if s in ds: r += 1
                else: c += 1
                cells.append((r, c))
            yield cells
    valid = [p for p in paths() if all(grid[r][c] != -1 for r, c in p)]
    if not valid:
        return 0
    return max(sum(grid[r][c] for r, c in set(p) | set(q)) for p in valid for q in valid)

rng = random.Random(3)
for _ in range(400):
    n = rng.randint(1, 5)
    g = [[rng.choice([0, 1, 1, -1]) for _ in range(n)] for _ in range(n)]
    g[0][0] = g[n - 1][n - 1] = 1
    assert my_cherry_ans(g) == cherry_brute(g)
    m, k = rng.randint(1, 4), rng.randint(1, 4)
    h = [[rng.choice([0, 1, 1, -1]) for _ in range(k)] for _ in range(m)]
    h[0][0] = h[m - 1][k - 1] = 0
    assert cherry_rect_ans(h) == rect_brute(h)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "hai người đồng thời trên lưới":**

```python
@lru_cache(None)
def f(r1, c1, r2):
    c2 = r1 + c1 - r2
    if ra ngoài hoặc gai: return -inf
    g = grid[r1][c1] + (grid[r2][c2] if r1 != r2 else 0)
    if (r1, c1) == đích: return g
    return g + max(f(r1+1,c1,r2+1), f(r1+1,c1,r2), f(r1,c1+1,r2+1), f(r1,c1+1,r2))
return max(0, f(0, 0, 0))
```

**Checklist trước khi submit:**

- [ ] Không dùng tham lam hai lượt.
- [ ] Hai người luôn đi **cùng bước**; cùng ô ⇔ `r1 == r2`.
- [ ] Ô trùng chỉ cộng **một lần**.
- [ ] Trạng thái không hợp lệ = `-inf`, không phải `0`.
- [ ] Kết quả cuối `max(0, ...)` (không có đường → 0).
- [ ] Kiểm tra `n = 1`.
- [ ] `O(n^3)` với `n ≤ 50` — an toàn.